### Delta Lake Module — Usage Examples
O módulo delta fornece utilitários para trabalhar com Delta Lake de forma segura, performática e escalável.
Ele cobre operações essenciais como:
- MERGE (upsert)
- SCD2
- CDC (Change Data Capture)
- Criação e leitura de tabelas Delta
- Otimização (OPTIMIZE + ZORDER)
- Time travel e histórico
- Vacuum e retenção

### Import Libraries
-----

In [ ]:
from pyspark.sql import SparkSession
from pyspark_utils.cleaning import (
    column_names,
    nulls,
    deduplication,
    dates,
    type_casting,
    text_cleaning
)

### Exemplo de Dataset
---

In [ ]:
updates = [
    ("POL123", "ACTIVE", 1000.0),
    ("POL456", "CANCELLED", 2000.0),
]
columns = ["policy_number", "status", "premium"]

df_updates = spark.createDataFrame(updates, columns)


### MERGE (Upsert)

In [ ]:
from pyspark_utils.delta import merge

merge.merge_upsert(
    spark,
    df_updates,
    target_path="/mnt/curated/policies",
    keys=["policy_number"]
)

'''
Quando usar:  
Atualizações incrementais de apólices, clientes, sinistros, produtos etc.
'''

### SCD2 (Slowly Changing Dimension Type 2)

In [ ]:
merge.merge_scd2(
    spark,
    df_updates,
    target_path="/mnt/curated/policies_scd2",
    keys=["policy_number"]
)

'''
Quando usar:  
Histórico de versões de apólices, endossos, alterações de clientes, pricing, underwriting.
'''

### CDC (Change Data Capture)

In [ ]:
from pyspark_utils.delta import CDC_processing

CDC_processing.apply_cdc(
    spark,
    df_cdc,
    target_path="/mnt/curated/policies",
    key="policy_number"
)


'''
Suporta:
    I = Insert
    U = Update
    D = Delete
'''

### Soft Delete

In [ ]:
CDC_processing.apply_soft_delete(
    spark,
    df_cdc,
    target_path="/mnt/curated/policies",
    key="policy_number",
    deleted_flag_col="is_deleted"
)

'''
Quando usar:  
Registros que não podem ser removidos fisicamente (compliance, auditoria).
'''


### CDC + SCD2

In [ ]:
CDC_processing.apply_cdc_scd2(
    spark,
    df_cdc,
    target_path="/mnt/curated/policies_scd2",
    key="policy_number"
)

'''
Quando usar:  
Histórico completo + operações incrementais.
'''

### Criar ou Substituir Tabela Delta

In [ ]:
from pyspark_utils.delta import tables

tables.create_or_replace_table(
    df_clean,
    path="/mnt/curated/policies",
    partition_by="policy_year"
)


### Ler Tabela Delta

In [ ]:
df = tables.read_table(spark, "/mnt/curated/policies")


### OPTIMIZE & ZORDER

In [ ]:
from pyspark_utils.delta import optimize

optimize.optimize_table(spark, "/mnt/curated/claims")

optimize.optimize_zorder(
    spark,
    "/mnt/curated/claims",
    ["policy_number"]
)

'''
Quando usar:
    - Tabelas grandes
    - Queries filtrando por colunas específicas
    - Melhorar performance de joins
'''

### Time Travel & History

**Ver histórico**

In [ ]:
from pyspark_utils.delta import history

history.show_history(spark, "/mnt/curated/policies")


**Ler versão específica**

In [ ]:
df_old = history.read_version(
    spark,
    "/mnt/curated/policies",
    version=3
)


### Vacuum (Limpeza de Arquivos Antigos)

In [ ]:
from pyspark_utils.delta import vacuum

vacuum.vacuum_table(
    spark,
    "/mnt/curated/policies",
    retention_hours=72
)
'''
Quando usar:
    - Reduzir custo de armazenamento
    - Manter apenas versões recentes
    - Após grandes merges ou deletes
'''